# 14 · Einsum, rolling windows, and repeated indices

**Level:** Expert practice · **Practice time:** 30–50 minutes

[← Previous](13_numerical_stability.ipynb) · [Course guide](../README.md) · [Next →](15_signal_processing_with_fft.ipynb)

**You will learn:** Express tensor contractions, calculate rolling features, and accumulate repeated indices.

**Prerequisites:** Notebooks 00–13.

## How to learn with this notebook

1. Run cells from top to bottom. Predict results before executing each example.
2. Do the **Try it** experiment by editing the example or adding a cell below it.
3. Replace `None` in each exercise with your answer, then rerun its check cell.
4. Consult the solutions at the end only after trying. An untouched exercise prints **Pending**, so Run All succeeds without claiming you solved it.
5. Complete the independent challenge, then restart the kernel and run all cells again.

Each notebook imports its own dependencies and creates its own data. You do not need variables from another notebook. The supplied outputs show a verified reference run; rerun cells to explore your changes.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
print("NumPy version:", np.__version__)

NumPy version: 2.5.3


## Step 1 · Einsum names the axes

`einsum` expresses sums of products using axis labels. In `ij,jk->ik`, j appears in both inputs and is absent from the output, so it is summed. Labels in the output survive. Start with an explicit output expression and compare with familiar operations before using more complex contractions.

In [2]:
A = np.arange(6, dtype=float).reshape(2, 3)
B = np.arange(12, dtype=float).reshape(3, 4)
product = np.einsum("ij,jk->ik", A, B, optimize=True)
row_energy = np.einsum("ij,ij->i", A, A)
print(product)
print("Row energy:", row_energy)
np.testing.assert_allclose(product, A @ B)
np.testing.assert_allclose(row_energy, (A * A).sum(axis=1))

[[20. 23. 26. 29.]
 [56. 68. 80. 92.]]
Row energy: [ 5. 50.]


**Try it:** Express the dot product of two 1D arrays and the trace of a square matrix with einsum.

## Step 2 · Rolling windows as views

`sliding_window_view` exposes overlapping windows without copying every window. By default the view is read-only because windows overlap. A window of width w on n values gives n-w+1 valid windows. Reducing all windows still performs work proportional to the number of window entries; for large rolling sums, cumulative sums can be cheaper.

In [3]:
from numpy.lib.stride_tricks import sliding_window_view
signal = np.array([1., 2., 3., 4., 5., 6.])
windows = sliding_window_view(signal, window_shape=3)
rolling_mean = windows.mean(axis=-1)
print(windows)
print("Rolling means:", rolling_mean)
assert windows.shape == (4, 3)
assert not windows.flags.writeable
assert np.shares_memory(signal, windows)
np.testing.assert_allclose(rolling_mean, [2, 3, 4, 5])

[[1. 2. 3.]
 [2. 3. 4.]
 [3. 4. 5.]
 [4. 5. 6.]]
Rolling means: [2. 3. 4. 5.]


**Try it:** Compute rolling minima and maxima, then align each result with the last index in its window.

## Step 3 · Repeated indices need unbuffered accumulation

`a[index] += value` uses buffered advanced indexing. Repeated positions may be updated only once rather than once per occurrence. `np.add.at` accumulates every occurrence. For nonnegative integer codes, bincount can be a simpler and often faster sum operation.

In [4]:
indices = np.array([0, 0, 2])
values = np.array([1., 1., 5.])
buffered = np.zeros(3)
buffered[indices] += values
accumulated = np.zeros(3)
np.add.at(accumulated, indices, values)
print("Buffered:", buffered)
print("Accumulated:", accumulated)
np.testing.assert_allclose(accumulated, [2, 0, 5])
np.testing.assert_allclose(accumulated, np.bincount(indices, weights=values, minlength=3))

Buffered: [1. 0. 5.]
Accumulated: [2. 0. 5.]


**Try it:** Use np.maximum.at to compute the largest value per group, starting from -infinity.

## Your practice exercises

Use the named answer variables. The checks verify your result; they do not require the same approach as the reference solution. If an assertion fails, read the expected value and inspect your shape and dtype.

### Exercise 1

Use einsum to compute one dot product per pair of corresponding rows.

In [5]:
left = np.array([[1., 2.], [3., 4.]])
right = np.array([[5., 6.], [7., 8.]])

row_dots = None  # TODO: replace None with your answer

In [6]:
if row_dots is None:
    print("Pending: complete Exercise 1 first.")
else:
    np.testing.assert_allclose(row_dots, [17, 53])
    print("Exercise 1 passed!")

Pending: complete Exercise 1 first.


### Exercise 2

Calculate valid rolling means with window width 2.

In [7]:
series = np.array([2., 4., 8., 10.])

rolling = None  # TODO: replace None with your answer

In [8]:
if rolling is None:
    print("Pending: complete Exercise 2 first.")
else:
    np.testing.assert_allclose(rolling, [3, 6, 9])
    print("Exercise 2 passed!")

Pending: complete Exercise 2 first.


## Independent challenge

Create a batch shaped (4, 3, 2). Compute each batch member’s 2×2 Gram matrix using both swapaxes(..., -1, -2) @ data and einsum, then compare.

Add code and explanation cells below. Decide on expected results before checking your implementation.

In [9]:
# Your independent challenge work goes here.

## Before you move on

- Explain the main idea in your own words without reading the examples.
- Make both exercise checks print **passed**.
- Try at least one changed input or edge case and explain the result.
- Finish the independent challenge or write down a specific question to revisit.

Expertise comes from repeated practice and debugging unfamiliar problems. Completing these notebooks gives you a broad practical foundation, not a substitute for that experience.

---

## Reference solutions — try first

These cells use separate reference variables, so they do not fill in your exercise answers. They also check the reference result. Compare your reasoning as well as the output.

### Solution 1

Use einsum to compute one dot product per pair of corresponding rows.

**Why it works:** The label i survives as the row axis. The repeated j label is summed, giving a dot product for each corresponding row pair.

In [10]:
reference_row_dots = np.einsum("ij,ij->i", left, right)
np.testing.assert_allclose(reference_row_dots, [17, 53])
print(reference_row_dots)

[17. 53.]


### Solution 2

Calculate valid rolling means with window width 2.

**Why it works:** The window view exposes adjacent pairs. Averaging its final axis produces one mean per complete pair.

In [11]:
reference_rolling = sliding_window_view(series, 2).mean(axis=-1)
np.testing.assert_allclose(reference_rolling, [3, 6, 9])
print(reference_rolling)

[3. 6. 9.]


## Continue learning

- [Official NumPy documentation: numpy.einsum ](https://numpy.org/doc/stable/reference/generated/numpy.einsum.html)
- [Official NumPy documentation: numpy.lib.stride_tricks.sliding_window_view ](https://numpy.org/doc/stable/reference/generated/numpy.lib.stride_tricks.sliding_window_view.html)
- [Official NumPy documentation: numpy.ufunc.at ](https://numpy.org/doc/stable/reference/generated/numpy.ufunc.at.html)

[Next →](15_signal_processing_with_fft.ipynb)